# UC-02: Invoice Three-Way Match — Feature Engineering

This notebook builds the complete feature matrix for UC-02. Each feature group is
explained with markdown, computed, and validated.

**Feature Groups:**
1. Vendor Composite Profile (shared feature store)
2. Vendor Historical Performance (shared feature store)
3. Vendor Invoice Behavior — Leave-One-Out (shared, with LOO)
4. Price Benchmarks (shared feature store)
5. UC-02 Specific Features

**Leakage Warning:** The following columns are EXCLUDED:
- `price_variance`, `quantity_variance` — ARE the label
- `payment_block`, `block_reason`, `status` — derived from label
- `unit_price_invoiced`, `quantity_invoiced`, invoice amounts — Mode A (pre-receipt)

## 1. Feature Groups Overview

| Group | Source | Features | Leakage Risk |
|-------|--------|----------|--------------|
| Vendor Composite Profile | vendor_master, vendor_category | v_preferred, v_status_encoded, v_country_risk, ... | None |
| Vendor Historical Performance | po_header, po_line_item, gr_* | v_total_po_count, v_rejection_rate, ... | None |
| Vendor Invoice Behavior | invoice_header, invoice_line_item | v_invoice_match_rate, v_price_variance_rate | **HIGH** — needs LOO |
| Price Benchmarks | po_line_item, material_master, contract_item | p_avg_unit_price, p_price_to_standard_ratio | None |
| UC-02 Specific | Derived from joins | uc02_is_contract_po, uc02_gr_has_rejection, ... | None |

## 2. Load Preprocessed Data

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

from ml.common.db_config import load_tables
from ml.common.feature_store import (
    compute_vendor_composite_profile,
    compute_vendor_historical_performance,
    compute_vendor_invoice_behavior_loo,
    compute_price_benchmarks,
)
from ml.common.utils import (
    CRITICALITY_MAP, CONFIDENTIALITY_MAP, convert_to_usd, encode_ordinal,
)
from ml.data_processing.python.uc02_preprocessing import (
    UC02_TABLES, build_uc02_base_dataset, add_temporal_features,
)

In [ ]:
# Load tables and build base dataset
csv_dir = project_root / "output" / "csv"
tables = load_tables("csv", UC02_TABLES, csv_dir=csv_dir)

df = build_uc02_base_dataset(tables)
df = add_temporal_features(df)
print(f"Base dataset: {df.shape[0]} rows, {df.shape[1]} columns")

## 3. Vendor Composite Profile

Static vendor attributes encoded for ML:
- **v_type_\*** — one-hot encoded vendor type (OEM, DISTRIBUTOR, etc.)
- **v_preferred** — 1 if preferred vendor
- **v_status_encoded** — ordinal (BLOCKED=0, CONDITIONAL=1, ACTIVE=2)
- **v_country_risk** — ordinal country risk tier (JP/DE/US=1, ..., VN=4)
- **v_payment_terms_days** — payment terms in days
- **v_has_early_discount** — 1 if 2/10NET30
- **v_category_count** — number of categories the vendor serves

In [ ]:
vendor_profile = compute_vendor_composite_profile(
    tables["vendor_master"], tables.get("vendor_category")
)
print(f"Vendor profile: {vendor_profile.shape}")
vendor_profile.head()

In [ ]:
df = df.merge(vendor_profile, on="vendor_id", how="left")
print(f"After vendor profile merge: {df.shape}")

## 4. Vendor Historical Performance

Transaction-derived vendor statistics:
- **v_total_po_count** — total POs placed with this vendor
- **v_total_po_value** — total PO value
- **v_on_time_delivery_actual** — fraction of deliveries on time
- **v_rejection_rate** — fraction of received quantities rejected
- **v_distinct_materials** — number of unique materials ordered

In [ ]:
vendor_perf = compute_vendor_historical_performance(
    tables["po_header"], tables["po_line_item"],
    tables.get("gr_header"), tables.get("gr_line_item"),
)
print(f"Vendor performance: {vendor_perf.shape}")
vendor_perf.describe()

In [ ]:
df = df.merge(vendor_perf, on="vendor_id", how="left")
print(f"After vendor performance merge: {df.shape}")

## 5. Vendor Invoice Behavior — Leave-One-Out

**Why LOO?** Vendor invoice match rate is the strongest predictor of future match
outcomes. But if we compute it using ALL invoices (including the one we're predicting),
we introduce target leakage — the model sees information it wouldn't have at prediction time.

**LOO Strategy:**
- For each invoice, compute vendor stats EXCLUDING that invoice
- Vendors with only 1 invoice get the global average (cold-start handling)
- This prevents the target from leaking into features

**Features:**
- **v_invoice_match_rate_loo** — vendor's historical full match rate (excluding this invoice)
- **v_price_variance_rate_loo** — vendor's price variance rate (excluding this invoice)
- **v_qty_variance_rate_loo** — vendor's quantity variance rate (excluding this invoice)
- **v_avg_price_variance_pct_loo** — average price variance magnitude

In [ ]:
%%time
vendor_inv_loo = compute_vendor_invoice_behavior_loo(
    tables["invoice_header"], tables.get("invoice_line_item")
)
print(f"Vendor invoice behavior LOO: {vendor_inv_loo.shape}")
vendor_inv_loo.describe()

In [ ]:
df = df.merge(vendor_inv_loo, on="invoice_id", how="left")
print(f"After vendor invoice LOO merge: {df.shape}")

## 6. Price Benchmarks

Material-level price statistics and contract price comparison:
- **p_avg_unit_price** — average unit price for this material across all POs
- **p_std_unit_price** — standard deviation of unit price
- **p_price_to_standard_ratio** — PO price / material standard cost
- **p_contract_price** — best (min) contract price for this material

Missing contract prices (off-contract items) → filled with 0 + a missing indicator.

In [ ]:
price_bench = compute_price_benchmarks(
    tables["po_line_item"], tables["material_master"],
    tables.get("contract_item"),
)

merge_keys = ["po_id", "po_line_number", "material_id"]
price_cols = [c for c in price_bench.columns if c.startswith("p_")]
df = df.merge(price_bench[merge_keys + price_cols], on=merge_keys, how="left")

# Missing indicator for contract price
df["p_has_contract_price"] = df["p_contract_price"].notna().astype(int)
df["p_contract_price"] = df["p_contract_price"].fillna(0)

print(f"After price benchmarks merge: {df.shape}")

## 7. UC-02 Specific Features

Features derived from the joined base dataset specific to the invoice matching problem:

| Feature | Computation | Rationale |
|---------|-------------|----------|
| `uc02_is_contract_po` | 1 if contract_id not null | Contract POs have agreed prices |
| `uc02_gr_qty_vs_po_qty` | gr_qty / po_qty | Delivery quantity discrepancies |
| `uc02_gr_has_rejection` | 1 if qty_rejected > 0 | Quality issues → price disputes |
| `uc02_days_gr_to_invoice` | (invoice_date - gr_date).days | Delay indicates complexity |
| `uc02_is_maverick_po` | maverick_flag | Non-compliant POs → higher variance |
| `uc02_po_type_rush` | 1 if po_type = 'RUSH' | Rush orders → less price negotiation |
| `uc02_price_vs_standard` | unit_price / standard_cost | Price deviation from standard |
| `uc02_delivery_delay_days` | (actual - requested).days | Late deliveries → disputes |
| `uc02_vendor_quality_score` | raw quality_score | Direct quality indicator |
| `uc02_vendor_risk_score` | raw risk_score | Risk correlates with issues |
| `uc02_vendor_otd_rate` | on_time_delivery_rate | Reliable vendors → fewer issues |
| `uc02_material_criticality` | ordinal (LOW=0, MED=1, HIGH=2) | Critical items → more scrutiny |
| `uc02_gr_status_quality_hold` | 1 if gr_status = 'QUALITY_HOLD' | Quality issues flag |
| `uc02_amount_usd` | PO total in USD | High-value POs → more oversight |
| `uc02_vendor_invoice_seq` | Sequential invoice # per vendor | Experience effect |

In [ ]:
from ml.uc_02_invoice_match.feature_engineering.feature_functions import (
    compute_uc02_specific_features, _compute_vendor_invoice_seq,
)

df = compute_uc02_specific_features(df)

# Vendor raw scores
vendor_scores = tables["vendor_master"][
    ["vendor_id", "quality_score", "risk_score", "on_time_delivery_rate", "esg_score"]
].rename(columns={
    "quality_score": "uc02_vendor_quality_score",
    "risk_score": "uc02_vendor_risk_score",
    "on_time_delivery_rate": "uc02_vendor_otd_rate",
    "esg_score": "uc02_vendor_esg_score",
})
df = df.merge(vendor_scores, on="vendor_id", how="left")

# Vendor invoice sequence
df["uc02_vendor_invoice_seq"] = _compute_vendor_invoice_seq(df)

print(f"After UC-02 features: {df.shape}")
uc02_cols = [c for c in df.columns if c.startswith("uc02_")]
print(f"UC-02 specific features: {uc02_cols}")

## 8. Label Engineering

**Binary target** (primary):
- 0 = FULL_MATCH
- 1 = ANY_VARIANCE (PRICE_VARIANCE | QUANTITY_VARIANCE | BOTH_VARIANCE)

**Multi-class target** (secondary):
- 0 = FULL_MATCH, 1 = PRICE_VARIANCE, 2 = QUANTITY_VARIANCE, 3 = BOTH_VARIANCE

In [ ]:
df["target_binary"] = (df["match_status"] != "FULL_MATCH").astype(int)
df["target_multiclass"] = df["match_status"].map({
    "FULL_MATCH": 0, "PRICE_VARIANCE": 1,
    "QUANTITY_VARIANCE": 2, "BOTH_VARIANCE": 3,
})

print("Binary target:")
print(df["target_binary"].value_counts())
print("\nMulti-class target:")
print(df["target_multiclass"].value_counts())

## 9. Feature Matrix Assembly

In [ ]:
from ml.uc_02_invoice_match.feature_engineering.feature_functions import (
    prepare_feature_matrix, ID_COLUMNS, LEAKAGE_COLUMNS,
)

X, y = prepare_feature_matrix(df, target="binary")
print(f"Feature matrix: {X.shape}")
print(f"Target: {y.shape}, distribution: {dict(y.value_counts())}")
print(f"\nFeature columns ({len(X.columns)}):")
for col in sorted(X.columns):
    print(f"  {col}")

In [ ]:
# Leakage check: no feature should correlate > 0.90 with target
correlations = X.corrwith(y).abs().sort_values(ascending=False)
print("Top 10 feature-target correlations:")
print(correlations.head(10))

if correlations.max() > 0.90:
    print(f"\n⚠ WARNING: Max correlation {correlations.max():.3f} — possible leakage!")
    print(f"  Feature: {correlations.idxmax()}")
else:
    print(f"\n✓ Max correlation: {correlations.max():.3f} — no leakage detected")

In [ ]:
# Save feature matrix
output_dir = project_root / "ml" / "uc_02_invoice_match" / "feature_engineering"
feature_df_save = X.copy()
feature_df_save["target_binary"] = y.values
feature_df_save["invoice_id"] = df["invoice_id"].values
feature_df_save.to_csv(output_dir / "feature_matrix.csv", index=False)
print(f"Saved feature matrix to {output_dir / 'feature_matrix.csv'}")

## 10. Summary Statistics

In [ ]:
print("Feature statistics:")
X.describe().T.round(3)

In [ ]:
# Correlation with target — bar chart
fig, ax = plt.subplots(figsize=(10, max(6, len(correlations) * 0.3)))
correlations.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Feature-Target Correlation (absolute)")
ax.set_xlabel("Absolute Correlation")
plt.tight_layout()
plt.show()